In [23]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [29]:
class Value:
    
    def __init__(self, data, _children=(), _op='', label = ''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data,(self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1
        
    def __sub__(self, other):
        return self + (-other) 

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        
        return out

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')

        def _backward():
            self.grad = out.data * out.data
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        self.grad = 1.0
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        for node in reversed(topo):
            node._backward()

In [30]:
import random;

class Neuron:
    
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__ (self, x):
        outs = [n(x) for n in self.neurons]
        return outs

    def parameters(self):
        params = []
        for neurons in self.neurons:
            ps = neurons.parameters()
            params.extend(ps)
        return params

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i + 1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
        


In [34]:
n = MLP(3, [4,4,1])
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]





In [106]:
for k in range(20):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout[0]-ygt)**2 for ygt, yout in zip(ys, ypred))
    loss    

    # backward pass
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()
    
    # linear gradient
    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)


0 9.564883721407458e-07
1 9.564856929124e-07
2 9.564830136993097e-07
3 9.564803345008028e-07
4 9.564776553173329e-07
5 9.564749761489579e-07
6 9.564722969954534e-07
7 9.564696178568681e-07
8 9.564669387332845e-07
9 9.564642596242812e-07
10 9.564615805307022e-07
11 9.564589014517463e-07
12 9.564562223876648e-07
13 9.564535433388384e-07
14 9.564508643047164e-07
15 9.564481852855988e-07
16 9.564455062813104e-07
17 9.564428272919853e-07
18 9.564401483175342e-07
19 9.564374693584223e-07


In [107]:
ypred

[[Value(data=0.999620723314867)],
 [Value(data=-0.9996293625249019)],
 [Value(data=-0.9994282928208301)],
 [Value(data=0.9994097751031925)]]